# Course 3: Sentiment Analysis using Naive Bayes
In this tutorial you'll learn how to build a sentiment analysis model using Naive Bayes: 

* Train a naive bayes model on a sentiment analysis task
* Test using your model
* Compute ratios of positive words to negative words
* Error analysis
* Predict on your own text

## Table of Contents

- [Importing Functions and Data](#0)
- [1 - Process the Data](#1)
- [2 - Train your Model using Naive Bayes](#2)
- [3 - Test your Naive Bayes](#3)
- [4 - Filter words by Ratio of Positive to Negative Counts](#4)
- [5 - Error Analysis](#5)
- [6 - Predict with your own Text](#6)

<a name='0'></a>
## Import Libraries and Data

In [1]:
import pdb
from nltk.corpus import stopwords, twitter_samples
import numpy as np
import pandas as pd
import nltk
import string
from nltk.tokenize import TweetTokenizer
from os import getcwd
nltk.download()
nltk.download('stopwords')

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gabriel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Imported Libraries

Download the data needed current tutorials. Feel free to doanload all data sets from nltk corpora. For additional information feel free to check for documentation https://www.nltk.org/book/.

* to use entire data sets you will need to download it using:
```Python
nltk.download()
```

* stopwords: to run this notebook on your local computer, you will need to download it using:
```python
nltk.download('stopwords')
```

In [2]:
#set the file path
filePath = f"{getcwd()}"
nltk.data.path.append(filePath)

In [3]:
# choose one dataset from nltk corpora
from nltk.corpus import pros_cons

### Prepare the Data
* The pros_cons contains subsets of more than 20K reviews for each class.  

In [4]:
# read the pros and cons reviews
reviews_cons = pros_cons.raw('IntegratedCons.txt')
reviews_pros = pros_cons.raw('IntegratedPros.txt')
#reviews_pros

In [5]:
#remove begin,end tags <Pros>, <Cons>
reviews_cons = reviews_cons.replace('<Cons>','').replace('</Cons>','').strip()
reviews_pros = reviews_pros.replace('<Pros>','').replace('</Pros>','').strip()
reviews_cons_list = reviews_cons.split('\n')
reviews_pros_list = reviews_pros.split('\n')

* Train set 80% and test set 20%.


In [6]:
train_pos = reviews_pros_list[:20000]
test_pos = reviews_pros_list[20000:24000]
train_neg = reviews_cons_list[:20000]
test_neg = reviews_cons_list[20000:24000]
train_x = train_pos + train_neg 
test_x = test_pos + test_neg

* Create the numpy array of positive labels and negative labels.

In [7]:
# combine positive and negative labels
train_x = train_pos + train_neg 
test_x = test_pos + test_neg
train_y = np.append(np.ones((len(train_pos), 1)), np.zeros((len(train_neg), 1)), axis=0)
test_y = np.append(np.ones((len(test_pos), 1)), np.zeros((len(test_neg), 1)), axis=0)

In [8]:
# Print the shape train and test sets
print("train_y.shape = " + str(train_y.shape))
print("test_y.shape = " + str(test_y.shape))

train_y.shape = (40000, 1)
test_y.shape = (5875, 1)


* Create the function for processing the string/text:
    - tokenization.
    - remove stop words.
    - apply stemming.  

In [9]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer

def process_text(str):
    """Process text function.
    Inputs:
        text: a string containing a text
    Output:
        tokens_clean: a list of words containing the processed string

    """
    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    # remove hyperlinks    
    str = re.sub(r'https?://[^\s\n\r]+', '', str)
    # remove hashtags
    # only removing the hash # sign from the word
    str = re.sub(r'#', '', str)
    # tokenize tweets
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True,
                               reduce_len=True)
    str_tokens = tokenizer.tokenize(str)

    tokens_clean = []
    for word in str_tokens:
        if (word not in stopwords_english and  # remove stopwords
                word not in string.punctuation):  # remove punctuation
            stem_word = stemmer.stem(word)  # stemming word
            tokens_clean.append(stem_word)

    return tokens_clean

* Create the frequency dictionary function.  

In [10]:
def build_freqs(strs, ys):
    """
    Inputs: strs - a list of strings
            ys - an mx1 array with the label (0 or 1) for each word
    Outputs: freqs - a dictionary of pairs of each word and label
    """
    yslist = np.squeeze(ys).tolist()
    # initialize an empty dictionary
    freqs={}
    for y, str in zip(yslist, strs):
        for word in process_text(str):
            pair = (word, y)
            if pair in freqs:
                freqs[pair] += 1
            else:
                freqs[pair] = 1
    return freqs

In [11]:
def check_freq(freqs, word, label):
    '''
    Inputs:
        freqs: a dictionary with the frequency of each pair (or tuple)
        word: the word to check
        label: the label corresponding to the word
    Outputs:
        n: the number of times the word with its corresponding label appears.
    '''
    n = 0  # freqs.get((word, label), 0)

    pair = (word, label)
    if (pair in freqs):
        n = freqs[pair]

    return n

In [12]:
# create frequency dictionary
freqs = build_freqs(train_x, train_y)
# check the output
print("type(freqs) = " + str(type(freqs)))
print("len(freqs) = " + str(len(freqs.keys())))

type(freqs) = <class 'dict'>
len(freqs) = 10361


### Process Text
The given function 'process_text' tokenizes the sentence into individual words, removes stop words and applies stemming.

In [13]:
# test the function below
print('This is an example of a positive text: \n', train_x[0])
print('\nThis is an example of the processed version of the text: \n', process_text(train_x[0]))

This is an example of a positive text: 
 Easy to use, economical!

This is an example of the processed version of the text: 
 ['easi', 'use', 'econom']


<a name='2'></a>
## 2 - Train your Model using Naive Bayes

Naive bayes is an algorithm that could be used for sentiment analysis. It takes a short time to train and also has a short prediction time.

#### So how do you train a Naive Bayes classifier?
- The first part of training a naive bayes classifier is to identify the number of classes that you have.
- You will create a probability for each class.
$P(D_{pos})$ is the probability that the document is positive.
$P(D_{neg})$ is the probability that the document is negative.
Use the formulas as follows and store the values in a dictionary:

$$P(D_{pos}) = \frac{D_{pos}}{D}\tag{1}$$

$$P(D_{neg}) = \frac{D_{neg}}{D}\tag{2}$$

Where $D$ is the total number of documents, or tweets in this case, $D_{pos}$ is the total number of positive tweets and $D_{neg}$ is the total number of negative tweets.

#### Prior and Logprior

The prior probability represents the underlying probability in the target population that a tweet is positive versus negative.  In other words, if we had no specific information and blindly picked a tweet out of the population set, what is the probability that it will be positive versus that it will be negative? That is the "prior".

The prior is the ratio of the probabilities $\frac{P(D_{pos})}{P(D_{neg})}$.
We can take the log of the prior to rescale it, and we'll call this the logprior

$$\text{logprior} = log \left( \frac{P(D_{pos})}{P(D_{neg})} \right) = log \left( \frac{D_{pos}}{D_{neg}} \right)$$.

Note that $log(\frac{A}{B})$ is the same as $log(A) - log(B)$.  So the logprior can also be calculated as the difference between two logs:

$$\text{logprior} = \log (P(D_{pos})) - \log (P(D_{neg})) = \log (D_{pos}) - \log (D_{neg})\tag{3}$$

#### Positive and Negative Probability of a Word
To compute the positive probability and the negative probability for a specific word in the vocabulary, we'll use the following inputs:

- $freq_{pos}$ and $freq_{neg}$ are the frequencies of that specific word in the positive or negative class. In other words, the positive frequency of a word is the number of times the word is counted with the label of 1.
- $N_{pos}$ and $N_{neg}$ are the total number of positive and negative words for all documents (for all tweets), respectively.
- $V$ is the number of unique words in the entire set of documents, for all classes, whether positive or negative.

We'll use these to compute the positive and negative probability for a specific word using this formula:

$$ P(W_{pos}) = \frac{freq_{pos} + 1}{N_{pos} + V}\tag{4} $$
$$ P(W_{neg}) = \frac{freq_{neg} + 1}{N_{neg} + V}\tag{5} $$

Notice that we add the "+1" in the numerator for additive smoothing.  This [wiki article](https://en.wikipedia.org/wiki/Additive_smoothing) explains more about additive smoothing.

#### Log likelihood
To compute the loglikelihood of that very same word, we can implement the following equations:

$$\text{loglikelihood} = \log \left(\frac{P(W_{pos})}{P(W_{neg})} \right)\tag{6}$$

In [14]:
def train_naive_bayes(freqs, train_x, train_y):
    '''
    Inputs:
        freqs: dictionary from (word, label)
        train_x: a list of sentences
        train_y: a list of labels (0,1)
    Outputs:
        logprior: the log prior
        loglikelihood: the log likelihood of you Naive bayes equation
    '''
    loglikelihood = {}
    logprior = 0

    # calculate V, the number of unique words in the vocabulary
    vocab = set([pair[0] for pair in freqs.keys()])
    V = len(vocab)    

    # calculate N_pos, N_neg, V_pos, V_neg
    N_pos = N_neg = V_pos=V_neg=0
    for pair in freqs.keys():
        # if the label is positive (greater than zero)
        if pair[1] > 0:

            # Increment the number of positive words by the count for this (word, label) pair
            N_pos += freqs[pair]
            V_pos += 1
        # else, the label is negative
        else:

            # increment the number of negative words by the count for this (word,label) pair
            N_neg += freqs[pair]
            V_neg += 1
    # Calculate D, the number of documents
    D = train_y.shape[0]

    # Calculate D_pos, the number of positive documents
    D_pos = train_y[train_y == 1].shape[0]

    # Calculate D_neg, the number of negative documents
    D_neg = train_y[train_y == 0].shape[0]

    # Calculate logprior
    logprior = np.log(D_pos / D) - np.log(D_neg / D)
    
    # For each word in the vocabulary...
    for word in vocab:
        # get the positive and negative frequency of the word
        freq_pos = freqs.get((word, 1), 0)
        freq_neg = freqs.get((word, 0), 0)

        # calculate the probability that each word is positive, and negative
        p_w_pos = (freq_pos + 1) / (N_pos + V)
        p_w_neg = (freq_neg + 1) / (N_neg + V)

        # calculate the log likelihood of the word
        loglikelihood[word] = np.log(p_w_pos / p_w_neg)

    return logprior, loglikelihood

In [15]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)
print(logprior)
print(len(loglikelihood))

0.0
7725


<a name='3'></a>
## 3 - Test your Naive Bayes

Now that we have the `logprior` and `loglikelihood`, we can test the naive bayes function by making predicting on some text!

<a name='ex-3'></a>
### Exercise 3 - naive_bayes_predict
Implement `naive_bayes_predict`.

**Instructions**:
Implement the `naive_bayes_predict` function to make predictions on text.
* The function takes in the `text`, `logprior`, `loglikelihood`.
* It returns the probability that the review belongs to the positive or negative class.
* For each review, sum up loglikelihoods of each word in the review.
* Also add the logprior to this sum to get the predicted sentiment of that review.

$$ p = logprior + \sum_i^N (loglikelihood_i)$$

#### Note
Note we calculate the prior from the training data, and that the training data is evenly split between positive and negative labels (4000 positive and 4000 negative reviews).  This means that the ratio of positive to negative 1, and the logprior is 0.

The value of 0.0 means that when we add the logprior to the log likelihood, we're just adding zero to the log likelihood.  However, please remember to include the logprior, because whenever the data is not perfectly balanced, the logprior will be a non-zero value.

In [16]:
def naive_bayes_predict(sentence, logprior, loglikelihood):
    '''
    Inputs:
        sentence: a string
        logprior: a number
        loglikelihood: a dictionary of words mapping to numbers
    Outputs:
        p: the sum of all the logliklihoods of each word in the tweet (if found in the dictionary) + logprior (a number)

    '''
    # process the sentence to get a list of words
    word_l = process_text(sentence)
    # initialize probability to zero
    p = 0
    # add the logprior
    p += logprior
    for word in word_l:
        # check if the word exists in the loglikelihood dictionary
        if word in loglikelihood:
            # add the log likelihood of that word to the probability
            p += loglikelihood[word]
    return p

In [17]:
my_tweet = 'Great picture quality. Easy to use. A lot of features'
p = naive_bayes_predict(my_tweet, logprior, loglikelihood)
print('The expected output is', p)

The expected output is 11.93116709324045


In [18]:
my_review = 'Low battery life, memory to small..'
p = naive_bayes_predict(my_review, logprior, loglikelihood)
print('The expected output is', p)

The expected output is -2.3122700330864063


In [19]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood):
    """
    Input:
        test_x: A list of sentences
        test_y: the corresponding labels for the list of sentences
        logprior: the logprior
        loglikelihood: a dictionary with the loglikelihoods for each word
    Output:
        accuracy: (# of sentences classified correctly)/(total # of sentences from test set)
    """
    accuracy = 0  # return this properly

    y_hats = []
    for s in test_x:
        # if the prediction is > 0
        if naive_bayes_predict(s, logprior, loglikelihood) > 0:
            # the predicted class is 1
            y_hat_i = 1
        else:
            # otherwise the predicted class is 0
            y_hat_i = 0

        # append the predicted class to the list y_hats
        y_hats.append(y_hat_i)

    # error is the average of the absolute values of the differences between y_hats and test_y
    error = np.mean(np.abs(np.asarray(y_hats) - test_y))

    # Accuracy is 1 minus the error
    accuracy = 1-error

    return accuracy

In [20]:
print("Naive Bayes accuracy = %0.4f" %
      (test_naive_bayes(test_x, test_y, logprior, loglikelihood)))

Naive Bayes accuracy = 0.5000


<a name='4'></a>
## 4 - Filter words by Ratio of Positive to Negative Counts

- Some words have more positive counts than others, and can be considered "more positive".  Likewise, some words can be considered more negative than others.
- One way for us to define the level of positiveness or negativeness, without calculating the log likelihood, is to compare the positive to negative frequency of the word.
    - Note that we can also use the log likelihood calculations to compare relative positivity or negativity of words.
- We can calculate the ratio of positive to negative frequencies of a word.
- Once we're able to calculate these ratios, we can also filter a subset of words that have a minimum ratio of positivity / negativity or higher.
- Similarly, we can also filter a subset of words that have a maximum ratio of positivity / negativity or lower (words that are at least as negative, or even more negative than a given threshold).


In [21]:
def get_ratio(freqs, word):
    '''
    Input:
        freqs: dictionary containing the words

    Output: a dictionary with keys 'positive', 'negative', and 'ratio'.
        Example: {'positive': 10, 'negative': 20, 'ratio': 0.5}
    '''
    pos_neg_ratio = {'positive': 0, 'negative': 0, 'ratio': 0.0}

    # use lookup() to find positive counts for the word (denoted by the integer 1)
    pos_neg_ratio['positive'] = check_freq(freqs, word, 1)
    
    # use lookup() to find negative counts for the word (denoted by integer 0)
    pos_neg_ratio['negative'] = check_freq(freqs, word, 0)
    
    # calculate the ratio of positive to negative counts for the word
    pos_neg_ratio['ratio'] = (pos_neg_ratio['positive'] + 1) / (pos_neg_ratio['negative'] + 1)

    return pos_neg_ratio

In [22]:
get_ratio(freqs, 'good')

{'positive': 3449, 'negative': 329, 'ratio': 10.454545454545455}

In [23]:
get_ratio(freqs, 'happi')

{'positive': 5, 'negative': 9, 'ratio': 0.6}

<a name='5'></a>
## 5 - Predict with your own text

In [25]:
# Feel free to change the tweet below
my_text = 'The resolution of my camera is bad! I do not like it ... bad bad bad!'
my_text = 'The resolution of my camera is nice! I do like it !'
#my_tweet = 'Great picture quality. Easy to use. A lot of features'
p = naive_bayes_predict(my_text, logprior, loglikelihood)
print(p)
if p > 0:
    print('Positive sentiment')
else: 
    print('Negative sentiment')

3.561667507340342
Positive sentiment
